In [ ]:
from pathlib import Path

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

05 Agriculture NDVI to RDF


This notebook enriches agricultural polygons with vegetation information derived from an NDVI raster. Instead of converting every raster pixel into RDF resources, zonal statistics are calculated for each agricultural area. The resulting attributes are then integrated into the RDF knowledge graph.

This approach keeps the graph compact while preserving meaningful semantic information about vegetation conditions.

Workflow:
Load the NDVI raster.
Load the agricultural polygons.
Reproject datasets if required.
Calculate zonal statistics (e.g., mean NDVI) for each polygon.
Add the calculated values to the agricultural dataset.
Generate RDF triples including the NDVI attributes.
Serialize the graph as Turtle (.ttl).
Output

The notebook produces:

an enriched agricultural dataset,
RDF triples containing NDVI information,
a Turtle file that extends the existing Agro Knowledge Graph.

In [ ]:
import geopandas as gpd
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS, XSD
import rasterio
from rasterio.mask import mask
from rasterstats import zonal_stats


In [ ]:
# import data / file paths

BASE_DIR = Path.cwd().parent  

OUTPUT_DIR = BASE_DIR / "data" / "processed"

INPUT_FILE_AGRAR = BASE_DIR / "data" / "processed" / "examples" / "agrar_test.geojson"

INPUT_FILE_NDVI = BASE_DIR / "data" / "processed" / "examples" / "ndvi_clipped.TIF"

INPUT_FILE_RDF = BASE_DIR / "data" / "processed" / "agro_individuals_with_geom.ttl"

OUTPUT_FILE = BASE_DIR / "data" / "processed" / "agro_individuals_with_geom_and_ndvi.ttl"



In [ ]:
# Set CRS
gdf = gpd.read_file(INPUT_FILE_AGRAR)

if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs(4326)

In [ ]:
# import Rasterfile and check CRS

with rasterio.open(INPUT_FILE_NDVI) as src:
    print(src.crs)


In [ ]:
# Set CRS to UTM for calculation

gdf_utm = gdf.to_crs(25832)

In [ ]:
# Mask (clip) the grid based on the geometry of each agricultural area
for idx, row in gdf_utm.iterrows():
    geom = [row.geometry]

    with rasterio.open(INPUT_FILE_NDVI) as src:
        out_image, out_transform = mask(src, geom, crop=True)

  

In [ ]:
# # Calculate mean NDVI values and add them to the GeoDataFrame

stats = zonal_stats(
    gdf_utm,
    INPUT_FILE_NDVI,
    stats=["mean"],
    nodata=-9999  
)

gdf_utm["mean_ndvi"] = [s["mean"] for s in stats]

In [ ]:
gdf_utm.head(2)

In [ ]:
# import RDF from notebook 03_agricultural_fields_to_rdf

g = Graph()
g.parse(INPUT_FILE_RDF, format="turtle")

In [ ]:
# Namespaces
AGRO = Namespace("http://www.semanticweb.org/frank/ontologies/2025/7/agro_geokg#")

g.add((AGRO.meanNDVI, RDF.type, RDF.Property))
g.add((AGRO.meanNDVI, RDFS.label, Literal("Mean NDVI", lang="en")))
g.add((AGRO.meanNDVI, RDFS.label, Literal("Mittlerer NDVI", lang="de")))

In [ ]:
# Enrich the existing RDF graph with mean NDVI values
for idx, row in gdf_utm.iterrows():

    agriculture_uri = AGRO[f"agriculture_{idx+1}"]

    g.add((
        agriculture_uri,
        AGRO.meanNDVI,
        Literal(float(row["mean_ndvi"]), datatype=XSD.double)
    ))

In [ ]:
g.serialize(destination=OUTPUT_FILE, format="turtle")

print("Features:", len(gdf))
print("Triples:", len(g))
print("Saved:", OUTPUT_FILE)